# Successive Cancellation List

In [1]:
import itertools
import logging
import random
from concurrent.futures import ProcessPoolExecutor
from wrappers.polar_wrapper import (
    polar_code_p2, get_logical_error_on_accepted_states, divide_half_list,
    get_q1prep_accepted_states,
)
from wrappers.stim_wrapper import (
    simulate_stim_polar_code_normal,
    
    calculate_logical_error_result_polar_normal,
    simulate_batch_and_save_result_polar_normal,
    generate_qiskit_polar_code, compiled_to_qiskit_hardware,
    find_and_delete_files
)

import pandas as pd
import os
import glob
import sys

from qiskit_ibm_runtime import QiskitRuntimeService

import collections
import numpy as np

In [2]:
# Import your compiled Cython wrappers
from Encoders.polar import PyEncoderPolar
# Adjust the import below if your Decoder wrapper class is named differently
from Decoders.SCL import PyDecoderPolarSCL 

In [ ]:
n = 4
lstate = "X"
sim_type = "normal"
i = 2
p_error = 0
shots = 1e4
seed = 1234

results = simulate_stim_polar_code_normal(n, lstate, sim_type, i, p_error, shots, seed)

sum(results.values())

KeyError: slice(None, None, -1)

In [4]:
zpos_list = [-1, -1, 1, 3, 6, 7, 22, 15, 90, 31, 362]
zpos_list[n] = i-1

accepted_states, data_qubit_states = get_q1prep_accepted_states(n, lstate, results, zpos_list)
sum(data_qubit_states.values())

8048

In [57]:
import numpy as np

def calculate_logical_error_rate(data_qubit_states, decoder, p_error, K, expected_info_bits=None):
    """
    Decodes the accepted Stim measurements and calculates the Logical Error Rate (LER).
    
    Args:
        data_qubit_states (dict): e.g., {'00111001': 2, ...}
        decoder: The initialized PyDecoderPolarSCL object.
        p_error (float): Physical error rate used for LLR calculation.
        K (int): Number of information bits.
        expected_info_bits (list/array): The expected K bits (default is all 0s for |0>_L).
        
    Returns:
        total_trials (int): Total number of accepted measurements.
        logical_errors (int): Number of measurements that decoded incorrectly.
        ler (float): Logical Error Rate.
    """
    # By default, state preparation of |0> or |+> usually expects the K info bits to be 0
    if expected_info_bits is None:
        expected_info_bits = np.zeros(K, dtype=np.int32)
    else:
        expected_info_bits = np.array(expected_info_bits, dtype=np.int32)

    # Calculate LLR magnitude from the physical error rate
    if p_error == 0:
        llr_mag = 10.0 # High confidence for noiseless
    else:
        llr_mag = np.log((1 - p_error) / p_error)

    total_trials = 0
    logical_errors = 0

    print(f"{'Measurement':<12} | {'Count':<5} | {'Decoded V_K':<11} | {'Status'}")
    print("-" * 45)

    for bit_str, count in data_qubit_states.items():
        # 1. Convert string to NumPy array of integers
        stim_measurements = np.array([int(b) for b in bit_str])

        # 2. Map hard bits to soft LLRs
        # 0 -> +llr_mag, 1 -> -llr_mag
        Y_N_LLRs = np.where(stim_measurements == 0, llr_mag, -llr_mag).astype(np.float64)

        # 3. Decode
        V_K_hat = np.zeros(K, dtype=np.int32)
        decoder.decode(Y_N_LLRs, V_K_hat, 0) # frame_id = 0

        # 4. Check if the decoder successfully recovered the intended state
        if np.array_equal(V_K_hat, expected_info_bits):
            status = "✅ Pass"
        else:
            status = "❌ FAIL"
            logical_errors += count # Add the frequency of this error!

        total_trials += count
        
        # # Optional: Print the first few results just to see what's happening
        # print(f"{bit_str:<12} | {count:<5} | {str(V_K_hat):<11} | {status}")

    # Calculate final LER
    ler = logical_errors / total_trials if total_trials > 0 else 0.0

    print("-" * 45)
    print(f"Total Accepted Trials : {total_trials}")
    print(f"Logical Errors        : {logical_errors}")
    print(f"Logical Error Rate    : {ler:.6f}")

    return total_trials, logical_errors, ler

In [69]:
# --- Your Setup ---
n = 4
N = 2**n  
lstate = "X"
sim_type = "normal"
i = 2
p_error = 0
shots = 1e4
seed = 1234

K = 1
L = 6

frozen_bits_mask = [True] * N
frozen_bits_mask[i-2] = False

# frozen_bits_mask = [True] * (N - K) + [False] * K

p_error = 0.0001  # Adjust this to whatever your Stim circuit used!

results = simulate_stim_polar_code_normal(n, lstate, sim_type, i, p_error, shots, seed)
accepted_states, data_qubit_states = get_q1prep_accepted_states(n, lstate, results)

# Initialize the decoder
decoder = PyDecoderPolarSCL(K, N, L, frozen_bits_mask)

# --- Run the function ---
print("\n=== STARTING DECODING AND LER CALCULATION ===")
total, errors, ler = calculate_logical_error_rate(
    data_qubit_states=data_qubit_states, 
    decoder=decoder, 
    p_error=p_error, 
    K=K
)


=== STARTING DECODING AND LER CALCULATION ===
Measurement  | Count | Decoded V_K | Status
---------------------------------------------
---------------------------------------------
Total Accepted Trials : 7
Logical Errors        : 3
Logical Error Rate    : 0.428571


In [ ]:
K = 1
N = 2**n
L = 4
frozen_bits_mask = [True] * (N - K) + [False] * K
decoder = PyDecoderPolarSCL(K, N, L, frozen_bits_mask)


if p_error == 0:
    llr_mag = 10.0 # Arbitrary high confidence for noiseless simulation
else:
    llr_mag = np.log((1 - p_error) / p_error)

# Let's say this is the array of measurements you got from Stim
stim_measurements = np.array([0,0,1,1,1,0,0,1])

# 3. Map the 0s and 1s to +LLR and -LLR
# If bit == 0, it becomes +llr_mag. If bit == 1, it becomes -llr_mag.
Y_N_LLRs = np.where(stim_measurements == 0, llr_mag, -llr_mag).astype(np.float64)

print(f"Hard bits from Stim: {stim_measurements}")
print(f"Soft LLRs for SCL  : {np.round(Y_N_LLRs, 2)}")


# V_K_hat: Array to hold the decoded information bits
V_K_hat = np.zeros(K, dtype=np.int32)


# Pass the LLRs (Y_N) and the output array (V_K_hat) to the C++ decoder
frame_id = 0
decoder.decode(Y_N_LLRs, V_K_hat, frame_id)
V_K_hat


--- Decoding ---
Hard bits from Stim: [0 0 1 1 1 0 0 1]
Soft LLRs for SCL  : [ 6.91  6.91 -6.91 -6.91 -6.91  6.91  6.91 -6.91]


array([0], dtype=int32)